In [3]:
# IMPORT LIBRARY
from pathlib import Path
import polars as pl
import duckdb

# MENENTUKAN FOLDER PROYEK
PROJECT_DIR = Path("/home/jovyan/work")
DATA_DIR = PROJECT_DIR / "data" / "raw"
DATA_PATH = DATA_DIR / "flood_dataset.parquet"

# MEMERIKSA DATASET
print("=== PERSIAPAN DATASET ===")
print("Folder proyek :", PROJECT_DIR)
print("Folder data   :", DATA_DIR)
print("Dataset       :", DATA_PATH.name)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}"
    )

print("Status dataset: BERHASIL DITEMUKAN")
print(f"Ukuran file   : {DATA_PATH.stat().st_size / (1024**2):.2f} MB")

=== PERSIAPAN DATASET ===
Folder proyek : /home/jovyan/work
Folder data   : /home/jovyan/work/data/raw
Dataset       : flood_dataset.parquet
Status dataset: BERHASIL DITEMUKAN
Ukuran file   : 16.27 MB


In [4]:
# MEMBACA DATASET
df = pl.read_parquet(DATA_PATH)

# INFORMASI DATASET
print("=== INFORMASI DATASET ===")
print(f"Jumlah baris : {df.height:,}")
print(f"Jumlah kolom : {df.width}")

# STRUKTUR KOLOM
print("\n=== STRUKTUR KOLOM ===")
print(df.schema)

# LIMA DATA PERTAMA
print("\n=== 5 DATA PERTAMA ===")
print(df.head(5))

=== INFORMASI DATASET ===
Jumlah baris : 1,152,432
Jumlah kolom : 13

=== STRUKTUR KOLOM ===
Schema({'district_id': String, 'date': Datetime(time_unit='us', time_zone=None), 'year': Int32, 'rain_1d': Float64, 'rain_3d': Float64, 'rain_7d': Float64, 'rain_30d': Float64, 'disc_now': Float64, 'disc_7d_mean': Float64, 'disc_change_3d': Float64, 'month': Int32, 'coastal': Int64, 'flood_label': Float64})

=== 5 DATA PERTAMA ===
shape: (5, 13)
┌───────────────┬──────────────┬──────┬─────────┬───┬──────────────┬───────┬─────────┬─────────────┐
│ district_id   ┆ date         ┆ year ┆ rain_1d ┆ … ┆ disc_change_ ┆ month ┆ coastal ┆ flood_label │
│ ---           ┆ ---          ┆ ---  ┆ ---     ┆   ┆ 3d           ┆ ---   ┆ ---     ┆ ---         │
│ str           ┆ datetime[μs] ┆ i32  ┆ f64     ┆   ┆ ---          ┆ i32   ┆ i64     ┆ f64         │
│               ┆              ┆      ┆         ┆   ┆ f64          ┆       ┆         ┆             │
╞═══════════════╪══════════════╪══════╪═════════╪═══╪═

In [5]:
# PEMERIKSAAN MISSING VALUES

print("=== JUMLAH NILAI KOSONG ===")

missing_values = df.null_count()

print(missing_values)

# PEMERIKSAAN DATA DUPLIKAT

print("\n=== PEMERIKSAAN DUPLIKASI ===")

jumlah_baris = df.height
jumlah_duplikat = df.is_duplicated().sum()

persentase_duplikat = (
    jumlah_duplikat / jumlah_baris
) * 100

print(f"Jumlah seluruh baris : {jumlah_baris:,}")
print(f"Jumlah baris duplikat : {jumlah_duplikat:,}")
print(f"Persentase duplikat   : {persentase_duplikat:.2f}%")

=== JUMLAH NILAI KOSONG ===
shape: (1, 13)
┌─────────────┬──────┬──────┬─────────┬───┬────────────────┬───────┬─────────┬─────────────┐
│ district_id ┆ date ┆ year ┆ rain_1d ┆ … ┆ disc_change_3d ┆ month ┆ coastal ┆ flood_label │
│ ---         ┆ ---  ┆ ---  ┆ ---     ┆   ┆ ---            ┆ ---   ┆ ---     ┆ ---         │
│ u32         ┆ u32  ┆ u32  ┆ u32     ┆   ┆ u32            ┆ u32   ┆ u32     ┆ u32         │
╞═════════════╪══════╪══════╪═════════╪═══╪════════════════╪═══════╪═════════╪═════════════╡
│ 0           ┆ 0    ┆ 0    ┆ 0       ┆ … ┆ 0              ┆ 0     ┆ 0       ┆ 0           │
└─────────────┴──────┴──────┴─────────┴───┴────────────────┴───────┴─────────┴─────────────┘

=== PEMERIKSAAN DUPLIKASI ===
Jumlah seluruh baris : 1,152,432
Jumlah baris duplikat : 0
Persentase duplikat   : 0.00%


In [6]:
# STATISTIK DESKRIPTIF DATASET

print("=== STATISTIK DESKRIPTIF ===")

kolom_numerik = [
    "year",
    "month",
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "rain_30d",
    "disc_now",
    "disc_7d_mean",
    "disc_change_3d",
    "coastal",
    "flood_label"
]

print(df.select(kolom_numerik).describe())


# CAKUPAN WAKTU DATASET

print("\n=== CAKUPAN WAKTU ===")

print(
    df.select(
        pl.col("date").min().alias("Tanggal awal"),
        pl.col("date").max().alias("Tanggal akhir")
    )
)


# CAKUPAN WILAYAH

print("\n=== CAKUPAN WILAYAH ===")

jumlah_wilayah = df.select(
    pl.col("district_id").n_unique()
).item()

print(f"Jumlah kecamatan: {jumlah_wilayah:,}")

=== STATISTIK DESKRIPTIF ===
shape: (9, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ year      ┆ month     ┆ rain_1d   ┆ … ┆ disc_7d_m ┆ disc_chan ┆ coastal   ┆ flood_la │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ean       ┆ ge_3d     ┆ ---       ┆ bel      │
│ str       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ ---       ┆ ---       ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ f64       ┆ f64       ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ count     ┆ 1.152432e ┆ 1.152432e ┆ 1.152432e ┆ … ┆ 1.152432e ┆ 1.152432e ┆ 1.152432e ┆ 1.152432 │
│           ┆ 6         ┆ 6         ┆ 6         ┆   ┆ 6         ┆ 6         ┆ 6         ┆ e6       │
│ null_coun ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0      │
│ t         ┆           ┆           ┆          

In [7]:
# PEMERIKSAAN VALIDITAS DATA

print("=== VALIDASI TAHUN DAN BULAN ===")

tahun_tidak_sesuai = df.filter(
    pl.col("year") != pl.col("date").dt.year()
).height

bulan_tidak_sesuai = df.filter(
    pl.col("month") != pl.col("date").dt.month()
).height

bulan_tidak_valid = df.filter(
    (pl.col("month") < 1) | (pl.col("month") > 12)
).height

print(f"Tahun tidak sesuai tanggal : {tahun_tidak_sesuai:,}")
print(f"Bulan tidak sesuai tanggal : {bulan_tidak_sesuai:,}")
print(f"Nilai bulan tidak valid    : {bulan_tidak_valid:,}")


# PEMERIKSAAN CURAH HUJAN NEGATIF

print("\n=== PEMERIKSAAN CURAH HUJAN ===")

kolom_hujan = [
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "rain_30d"
]

for kolom in kolom_hujan:
    jumlah_negatif = df.filter(
        pl.col(kolom) < 0
    ).height

    print(f"{kolom}: {jumlah_negatif:,} nilai negatif")


# DISTRIBUSI LABEL BANJIR

print("\n=== DISTRIBUSI LABEL BANJIR ===")

print(
    df.group_by("flood_label")
    .len()
    .sort("flood_label")
)


# DISTRIBUSI WILAYAH PESISIR

print("\n=== DISTRIBUSI WILAYAH PESISIR ===")

print(
    df.group_by("coastal")
    .len()
    .sort("coastal")
)

=== VALIDASI TAHUN DAN BULAN ===
Tahun tidak sesuai tanggal : 0
Bulan tidak sesuai tanggal : 0
Nilai bulan tidak valid    : 0

=== PEMERIKSAAN CURAH HUJAN ===
rain_1d: 0 nilai negatif
rain_3d: 0 nilai negatif
rain_7d: 0 nilai negatif
rain_30d: 0 nilai negatif

=== DISTRIBUSI LABEL BANJIR ===
shape: (2, 2)
┌─────────────┬─────────┐
│ flood_label ┆ len     │
│ ---         ┆ ---     │
│ f64         ┆ u32     │
╞═════════════╪═════════╡
│ 0.0         ┆ 1068121 │
│ 1.0         ┆ 84311   │
└─────────────┴─────────┘

=== DISTRIBUSI WILAYAH PESISIR ===
shape: (2, 2)
┌─────────┬─────────┐
│ coastal ┆ len     │
│ ---     ┆ ---     │
│ i64     ┆ u32     │
╞═════════╪═════════╡
│ 0       ┆ 115968  │
│ 1       ┆ 1036464 │
└─────────┴─────────┘


In [8]:
# PEMERIKSAAN OUTLIER MENGGUNAKAN METODE IQR

kolom_outlier = [
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "rain_30d",
    "disc_now",
    "disc_7d_mean",
    "disc_change_3d"
]

hasil_outlier = []

for kolom in kolom_outlier:

    statistik = df.select(
        pl.col(kolom).quantile(0.25).alias("Q1"),
        pl.col(kolom).quantile(0.75).alias("Q3")
    )

    q1 = statistik["Q1"].item()
    q3 = statistik["Q3"].item()

    iqr = q3 - q1

    batas_bawah = q1 - (1.5 * iqr)
    batas_atas = q3 + (1.5 * iqr)

    jumlah_outlier = df.filter(
        (pl.col(kolom) < batas_bawah) |
        (pl.col(kolom) > batas_atas)
    ).height

    hasil_outlier.append({
        "kolom": kolom,
        "Q1": q1,
        "Q3": q3,
        "batas_bawah": batas_bawah,
        "batas_atas": batas_atas,
        "jumlah_outlier": jumlah_outlier,
        "persentase_outlier": (
            jumlah_outlier / df.height
        ) * 100
    })

print("=== HASIL PEMERIKSAAN OUTLIER ===")

print(pl.DataFrame(hasil_outlier))

=== HASIL PEMERIKSAAN OUTLIER ===
shape: (7, 7)
┌────────────────┬──────────┬───────────┬─────────────┬────────────┬───────────────┬───────────────┐
│ kolom          ┆ Q1       ┆ Q3        ┆ batas_bawah ┆ batas_atas ┆ jumlah_outlie ┆ persentase_ou │
│ ---            ┆ ---      ┆ ---       ┆ ---         ┆ ---        ┆ r             ┆ tlier         │
│ str            ┆ f64      ┆ f64       ┆ f64         ┆ f64        ┆ ---           ┆ ---           │
│                ┆          ┆           ┆             ┆            ┆ i64           ┆ f64           │
╞════════════════╪══════════╪═══════════╪═════════════╪════════════╪═══════════════╪═══════════════╡
│ rain_1d        ┆ 0.3      ┆ 9.8       ┆ -13.95      ┆ 24.05      ┆ 60717         ┆ 5.268597      │
│ rain_3d        ┆ 2.2      ┆ 30.9      ┆ -40.85      ┆ 73.95      ┆ 34406         ┆ 2.985512      │
│ rain_7d        ┆ 8.4      ┆ 72.2      ┆ -87.3       ┆ 167.9      ┆ 23955         ┆ 2.078648      │
│ rain_30d       ┆ 60.0     ┆ 310.2     ┆ -

In [9]:
# PEMERIKSAAN DUPLIKASI KUNCI WILAYAH DAN TANGGAL

print("=== DUPLIKASI DISTRICT_ID DAN DATE ===")

duplikat_kunci = (
    df.group_by(["district_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
)

print(f"Jumlah kombinasi wilayah-tanggal duplikat: {duplikat_kunci.height:,}")

print("\n=== CONTOH DATA DUPLIKAT ===")

print(duplikat_kunci.head(10))

=== DUPLIKASI DISTRICT_ID DAN DATE ===
Jumlah kombinasi wilayah-tanggal duplikat: 0

=== CONTOH DATA DUPLIKAT ===
shape: (0, 3)
┌─────────────┬──────────────┬─────┐
│ district_id ┆ date         ┆ len │
│ ---         ┆ ---          ┆ --- │
│ str         ┆ datetime[μs] ┆ u32 │
╞═════════════╪══════════════╪═════╡
└─────────────┴──────────────┴─────┘


In [10]:
# PEMERIKSAAN CAKUPAN DATA PER KECAMATAN

coverage = (
    df.group_by("district_id")
    .agg(
        pl.len().alias("jumlah_observasi"),
        pl.col("date").n_unique().alias("jumlah_tanggal"),
        pl.col("date").min().alias("tanggal_awal"),
        pl.col("date").max().alias("tanggal_akhir")
    )
    .sort("district_id")
)

print("=== CAKUPAN DATA PER KECAMATAN ===")
print(coverage.head(10))

print("\n=== STATISTIK JUMLAH OBSERVASI ===")
print(
    coverage.select(
        pl.col("jumlah_observasi").min().alias("Minimum"),
        pl.col("jumlah_observasi").max().alias("Maksimum"),
        pl.col("jumlah_observasi").mean().alias("Rata-rata")
    )
)

print("\n=== JUMLAH KECAMATAN ===")
print(coverage.height)

=== CAKUPAN DATA PER KECAMATAN ===
shape: (10, 5)
┌────────────────┬──────────────────┬────────────────┬─────────────────────┬─────────────────────┐
│ district_id    ┆ jumlah_observasi ┆ jumlah_tanggal ┆ tanggal_awal        ┆ tanggal_akhir       │
│ ---            ┆ ---              ┆ ---            ┆ ---                 ┆ ---                 │
│ str            ┆ u32              ┆ u32            ┆ datetime[μs]        ┆ datetime[μs]        │
╞════════════════╪══════════════════╪════════════════╪═════════════════════╪═════════════════════╡
│ IDN.10.13.10_1 ┆ 3624             ┆ 3624           ┆ 2015-01-30 00:00:00 ┆ 2024-12-31 00:00:00 │
│ IDN.10.13.11_1 ┆ 3624             ┆ 3624           ┆ 2015-01-30 00:00:00 ┆ 2024-12-31 00:00:00 │
│ IDN.10.13.12_1 ┆ 3624             ┆ 3624           ┆ 2015-01-30 00:00:00 ┆ 2024-12-31 00:00:00 │
│ IDN.10.13.13_1 ┆ 3624             ┆ 3624           ┆ 2015-01-30 00:00:00 ┆ 2024-12-31 00:00:00 │
│ IDN.10.13.14_1 ┆ 3624             ┆ 3624           ┆ 2015

In [11]:
# VALIDASI KELENGKAPAN TANGGAL

tanggal_awal = df["date"].min()
tanggal_akhir = df["date"].max()

jumlah_tanggal = df["date"].n_unique()

jumlah_hari_seharusnya = (
    (tanggal_akhir - tanggal_awal).days + 1
)

print("=== VALIDASI KELENGKAPAN TANGGAL ===")
print(f"Tanggal awal              : {tanggal_awal}")
print(f"Tanggal akhir             : {tanggal_akhir}")
print(f"Jumlah tanggal unik       : {jumlah_tanggal:,}")
print(f"Jumlah hari seharusnya    : {jumlah_hari_seharusnya:,}")

if jumlah_tanggal == jumlah_hari_seharusnya:
    print("\nStatus: SELURUH TANGGAL TERCAKUP")
else:
    print("\nStatus: TERDAPAT TANGGAL YANG TIDAK TERCATAT")

=== VALIDASI KELENGKAPAN TANGGAL ===
Tanggal awal              : 2015-01-30 00:00:00
Tanggal akhir             : 2024-12-31 00:00:00
Jumlah tanggal unik       : 3,624
Jumlah hari seharusnya    : 3,624

Status: SELURUH TANGGAL TERCAKUP
